In [17]:
from dataclasses import dataclass

# ===================== ENTIDADES =====================

@dataclass
class Cliente:
    id: int
    nome: str
    telefone: str

    def __str__(self):
        return f"{self.id} - {self.nome} ({self.telefone})"


@dataclass
class Pet:
    id: int
    nome: str
    especie: str
    id_dono: int

    def __str__(self):
        return f"{self.id} - {self.nome} ({self.especie})"


@dataclass
class Servico:
    id: int
    nome: str
    preco: float

    def __str__(self):
        return f"{self.id} - {self.nome} (R$ {self.preco:.2f})"


class Agendamento:
    def __init__(self, id, id_pet, id_servico, data):
        self.id = id
        self.id_pet = id_pet
        self.id_servico = id_servico
        self.data = data
        self.concluido = False

    def concluir(self):
        self.concluido = True

    def __str__(self):
        status = "Concluído" if self.concluido else "Pendente"
        return f"{self.id} - PetID: {self.id_pet} - ServicoID: {self.id_servico} - Data: {self.data} - {status}"


# ===================== SISTEMA =====================

class Petshop:
    def __init__(self):
        self.clientes = []
        self.pets = []
        self.servicos = [
            Servico(1, "Banho", 50.0),
            Servico(2, "Tosa", 70.0),
            Servico(3, "Banho + Tosa", 110.0),
        ]
        self.agendamentos = []

        self.prox_cliente_id = 1
        self.prox_pet_id = 1
        self.prox_agendamento_id = 1

    # ===================== BUSCAS =====================

    def buscar_cliente(self, id_cliente):
        return next((c for c in self.clientes if c.id == id_cliente), None)

    def buscar_pet(self, id_pet):
        return next((p for p in self.pets if p.id == id_pet), None)

    def buscar_servico(self, id_servico):
        return next((s for s in self.servicos if s.id == id_servico), None)

    def buscar_agendamento(self, id_ag):
        return next((a for a in self.agendamentos if a.id == id_ag), None)

    # ===================== CADASTROS =====================

    def cadastrar_cliente(self, nome, telefone):
        if not nome:
            print("Nome não pode ser vazio.")
            return

        cliente = Cliente(self.prox_cliente_id, nome, telefone)
        self.clientes.append(cliente)
        self.prox_cliente_id += 1

        print(f"Cliente cadastrado! ID: {cliente.id}")

    def cadastrar_pet(self, nome, especie, id_cliente):
        if not nome:
            print("Nome do pet não pode ser vazio.")
            return

        cliente = self.buscar_cliente(id_cliente)
        if not cliente:
            print("Cliente não encontrado.")
            return

        pet = Pet(self.prox_pet_id, nome, especie, id_cliente)
        self.pets.append(pet)
        self.prox_pet_id += 1

        print(f"Pet cadastrado! ID: {pet.id}")

    # ===================== AGENDAMENTO =====================

    def agendar_servico(self, id_pet, id_servico, data):
        pet = self.buscar_pet(id_pet)
        servico = self.buscar_servico(id_servico)

        if not pet:
            print("Pet não encontrado.")
            return

        if not servico:
            print("Serviço não encontrado.")
            return

        ag = Agendamento(self.prox_agendamento_id, id_pet, id_servico, data)
        self.agendamentos.append(ag)
        self.prox_agendamento_id += 1

        print(f"Agendamento criado! ID: {ag.id}")

    def concluir_agendamento(self, id_ag):
        ag = self.buscar_agendamento(id_ag)

        if not ag:
            print("Agendamento não encontrado.")
            return

        if ag.concluido:
            print("Já está concluído.")
            return

        ag.concluir()
        print("Agendamento concluído!")

    # ===================== LISTAGENS =====================

    def listar_clientes(self):
        if not self.clientes:
            print("Nenhum cliente cadastrado.")
            return

        for c in self.clientes:
            print(c)

    def listar_pets(self):
        if not self.pets:
            print("Nenhum pet cadastrado.")
            return

        for p in self.pets:
            dono = self.buscar_cliente(p.id_dono)
            nome_dono = dono.nome if dono else "Desconhecido"
            print(f"{p} - Dono: {nome_dono}")

    def listar_servicos(self):
        for s in self.servicos:
            print(s)

    def listar_agendamentos(self):
        if not self.agendamentos:
            print("Nenhum agendamento.")
            return

        for a in self.agendamentos:
            pet = self.buscar_pet(a.id_pet)
            servico = self.buscar_servico(a.id_servico)

            nome_pet = pet.nome if pet else "Desconhecido"
            nome_servico = servico.nome if servico else "Desconhecido"
            status = "Concluído" if a.concluido else "Pendente"

            print(f"{a.id} - Pet: {nome_pet} - Serviço: {nome_servico} - Data: {a.data} - {status}")

    # ===================== RELATÓRIO =====================

    def calcular_faturamento(self):
        total = 0.0
        for ag in self.agendamentos:
            if ag.concluido:
                servico = self.buscar_servico(ag.id_servico)
                if servico:
                    total += servico.preco
        return total


# ===================== UTIL =====================

def ler_int(msg):
    while True:
        try:
            return int(input(msg))
        except ValueError:
            print("Valor inválido.")


# ===================== MENU =====================

petshop = Petshop()

while True:
    print("\n=== Petshop ===")
    print("1 - Cadastrar cliente")
    print("2 - Listar clientes")
    print("3 - Cadastrar pet")
    print("4 - Listar pets")
    print("5 - Listar serviços")
    print("6 - Agendar serviço")
    print("7 - Listar agendamentos")
    print("8 - Concluir agendamento")
    print("9 - Ver faturamento")
    print("0 - Sair")

    op = input("Escolha: ")

    if op == "1":
        nome = input("Nome: ")
        tel = input("Telefone: ")
        petshop.cadastrar_cliente(nome, tel)

    elif op == "2":
        petshop.listar_clientes()

    elif op == "3":
        nome = input("Nome do pet: ")
        especie = input("Espécie: ")
        id_cliente = ler_int("ID do dono: ")
        petshop.cadastrar_pet(nome, especie, id_cliente)

    elif op == "4":
        petshop.listar_pets()

    elif op == "5":
        petshop.listar_servicos()

    elif op == "6":
        id_pet = ler_int("ID do pet: ")
        id_servico = ler_int("ID do serviço: ")
        data = input("Data: ")
        petshop.agendar_servico(id_pet, id_servico, data)

    elif op == "7":
        petshop.listar_agendamentos()

    elif op == "8":
        id_ag = ler_int("ID do agendamento: ")
        petshop.concluir_agendamento(id_ag)

    elif op == "9":
        total = petshop.calcular_faturamento()
        print(f"Faturamento total: R$ {total:.2f}")

    elif op == "0":
        break

    else:
        print("Opção inválida.")


=== Petshop ===
1 - Cadastrar cliente
2 - Listar clientes
3 - Cadastrar pet
4 - Listar pets
5 - Listar serviços
6 - Agendar serviço
7 - Listar agendamentos
8 - Concluir agendamento
9 - Ver faturamento
0 - Sair
Escolha: 5

1 - Banho (R$ 50.00)
2 - Tosa (R$ 70.00)
3 - Banho + Tosa (R$ 110.00)

=== Petshop ===
1 - Cadastrar cliente
2 - Listar clientes
3 - Cadastrar pet
4 - Listar pets
5 - Listar serviços
6 - Agendar serviço
7 - Listar agendamentos
8 - Concluir agendamento
9 - Ver faturamento
0 - Sair
Escolha: 6
ID do pet: 2
ID do serviço: 1
Data: 2

Pet ou serviço não encontrado.

=== Petshop ===
1 - Cadastrar cliente
2 - Listar clientes
3 - Cadastrar pet
4 - Listar pets
5 - Listar serviços
6 - Agendar serviço
7 - Listar agendamentos
8 - Concluir agendamento
9 - Ver faturamento
0 - Sair
Escolha: 1
Nome: miguel
Telefone: 123

Cliente cadastrado! ID: 1

=== Petshop ===
1 - Cadastrar cliente
2 - Listar clientes
3 - Cadastrar pet
4 - Listar pets
5 - Listar serviços
6 - Agendar serviço
7 - Li